# Reflection sensitivity

This notebook runs the full in-memory pipeline on a synthetic snapshot-style dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.geometry import UnitCell, cell_to_reciprocal_matrix, rotation_matrix_from_axis_angle
from src.pipeline import PipelineConfig, run_pipeline_from_tables
from src.visualization import plot_dynamical_score_distribution, plot_detector_map

In [ ]:
cell = UnitCell(a=15.0, b=15.0, c=15.0, alpha=90, beta=90, gamma=90, composition={'C': 24, 'N': 8, 'Zn': 4})
ub0 = cell_to_reciprocal_matrix(cell)
orientation_rows = []
reflection_rows = []
for frame in range(1, 11):
    rot = rotation_matrix_from_axis_angle([0, 1, 0], (frame - 1) * 2.0)
    ub = rot @ ub0
    orientation_rows.append({
        'frame': frame, 'phi': (frame - 1) * 2.0,
        'UB11': ub[0,0], 'UB12': ub[0,1], 'UB13': ub[0,2],
        'UB21': ub[1,0], 'UB22': ub[1,1], 'UB23': ub[1,2],
        'UB31': ub[2,0], 'UB32': ub[2,1], 'UB33': ub[2,2],
    })
    for hkl, x, y in [((1,0,0), 1000+frame*2, 1010), ((0,1,0), 1025, 995-frame*3), ((1,1,0), 980, 1030)]:
        reflection_rows.append({'frame': frame, 'h': hkl[0], 'k': hkl[1], 'l': hkl[2], 'I': 1000 + 50*np.cos(frame), 'sigma': 25 + frame, 'x': x, 'y': y})
orientations = pd.DataFrame(orientation_rows)
reflections = pd.DataFrame(reflection_rows)
results = run_pipeline_from_tables(orientations, reflections, cell, PipelineConfig(dmin=1.0, dmax=20.0, hkl_limit=10, alpha=0.4))
results.reflection_table.head()

In [ ]:
plot_dynamical_score_distribution(results.reflection_table);
plot_detector_map(results.reflection_table);
plt.show()